Given your setup:

```text id="dataset_layout"
Google Drive
│
└── FloorPlanDataset/
    │
    ├── train/
    │   ├── images/
    │   └── labels/
    │
    └── data.yaml
```

and you only have a **train** folder uploaded to Google Drive, the notebook should:

1. Mount Google Drive
2. Read images from `train/images`
3. Split into Train / Valid / Test
4. Create YOLO folder structure
5. Generate updated `data.yaml`
6. Train YOLOv5
7. Evaluate
8. Export model

# Cell 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Cell 2: Define Dataset Location

Update this path according to your Drive:

In [ ]:
DATASET_ROOT = "/content/drive/MyDrive/FloorPlanDataset"

Expected:
```text
FloorPlanDataset/
│
├── train/
│   ├── images/
│   └── labels/
│
└── data.yaml
```

# Cell 3: Clone YOLOv5

In [ ]:
!git clone https://github.com/ultralytics/yolov5

In [ ]:
%cd yolov5

In [ ]:
!pip install -qr requirements.txt

# Cell 4: Verify GPU

In [ ]:
import torch

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Cell 5: Create Train/Valid/Test Split

In [ ]:
import os
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split

ROOT = DATASET_ROOT

SOURCE_IMAGES = f"{ROOT}/train/images"
SOURCE_LABELS = f"{ROOT}/train/labels"

SPLIT_ROOT = f"{ROOT}/split_dataset"

# Create folders

In [ ]:
for split in ["train", "valid", "test"]:
    os.makedirs(f"{SPLIT_ROOT}/{split}/images", exist_ok=True)
    os.makedirs(f"{SPLIT_ROOT}/{split}/labels", exist_ok=True)

# Read images

In [ ]:
image_files = []

for ext in ["*.jpg","*.jpeg","*.png"]:
    image_files.extend(
        list(Path(SOURCE_IMAGES).glob(ext))
    )

print("Total Images:", len(image_files))

# 80 / 10 / 10

In [ ]:
train_imgs, temp_imgs = train_test_split(
    image_files,
    test_size=0.20,
    random_state=42
)

val_imgs, test_imgs = train_test_split(
    temp_imgs,
    test_size=0.50,
    random_state=42
)

splits = {
    "train": train_imgs,
    "valid": val_imgs,
    "test": test_imgs
}

for split_name, files in splits.items():

    for img_path in files:

        stem = img_path.stem

        label_path = Path(
            SOURCE_LABELS
        ) / f"{stem}.txt"

        shutil.copy2(
            img_path,
            f"{SPLIT_ROOT}/{split_name}/images/{img_path.name}"
        )

        if label_path.exists():

            shutil.copy2(
                label_path,
                f"{SPLIT_ROOT}/{split_name}/labels/{stem}.txt"
            )

print("Split Completed")

# Cell 6: Verify Split

In [ ]:
for split in ["train","valid","test"]:

    img_count = len(
        os.listdir(
            f"{SPLIT_ROOT}/{split}/images"
        )
    )

    lbl_count = len(
        os.listdir(
            f"{SPLIT_ROOT}/{split}/labels"
        )
    )

    print(split)
    print("Images:", img_count)
    print("Labels:", lbl_count)
    print("-"*30)

# Cell 7: Generate data.yaml

Using your uploaded classes:

In [ ]:
import yaml

data_yaml = {
    "train": f"{SPLIT_ROOT}/train/images",
    "val": f"{SPLIT_ROOT}/valid/images",
    "test": f"{SPLIT_ROOT}/test/images",
    "nc": 11,
    "names": [
        "bed",
        "dining_table",
        "door",
        "entry_door",
        "kitchen_counter",
        "north",
        "sofa",
        "storage",
        "tv",
        "wc",
        "window"
    ]
}

In [ ]:
yaml_path = f"{SPLIT_ROOT}/data.yaml"

with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f)

print("Created:", yaml_path)

# Cell 8: Validate Dataset

In [ ]:
!python utils/autoanchor.py

Optional sanity check:

In [ ]:
import glob

print("Train:",
      len(glob.glob(f"{SPLIT_ROOT}/train/images/*")))

print("Valid:",
      len(glob.glob(f"{SPLIT_ROOT}/valid/images/*")))

print("Test:",
      len(glob.glob(f"{SPLIT_ROOT}/test/images/*")))

# Cell 9: Start Training

In [ ]:
!python train.py \
--img 1024 \
--batch 8 \
--epochs 200 \
--data $yaml_path \
--weights yolov5m.pt \
--cache \
--patience 30 \
--workers 2 \
--name floorplan_detector

# Cell 10: View Training Results

In [ ]:
from IPython.display import Image

Image(
    "runs/train/floorplan_detector/results.png"
)

# Cell 11: Evaluate Best Model

In [ ]:
!python val.py \
--weights runs/train/floorplan_detector/weights/best.pt \
--data $yaml_path \
--img 1024

# Cell 12: Run Prediction

In [ ]:
TEST_IMAGE = f"{SPLIT_ROOT}/test/images"

!python detect.py \
--weights runs/train/floorplan_detector/weights/best.pt \
--source $TEST_IMAGE \
--img 1024 \
--conf 0.25

# Cell 13: Show Predictions

In [ ]:
from glob import glob
from IPython.display import Image

preds = glob("runs/detect/exp/*.jpg")

Image(preds[0])

# Cell 14: Export Model

### ONNX

In [ ]:
!python export.py \
--weights runs/train/floorplan_detector/weights/best.pt \
--include onnx

### TensorRT

In [ ]:
!python export.py \
--weights runs/train/floorplan_detector/weights/best.pt \
--include engine

## Recommended Training Settings for Your Floor Plan Dataset

| Parameter        | Value   |
| ---------------- | ------- |
| Model            | YOLOv5m |
| Image Size       | 1024    |
| Batch Size       | 8       |
| Epochs           | 200     |
| Patience         | 30      |
| Cache            | True    |
| Train Split      | 80%     |
| Validation Split | 10%     |
| Test Split       | 10%     |

If you have **fewer than 500 floor plans**, I would modify the notebook to use **90% train / 10% validation only** and skip the test split until the final model evaluation. This generally improves detector quality when data is limited.
